# Project 2, comparison run: detecting scratches on wafers (MixedWM38)

**Purpose.** This notebook repeats the pipeline of `Project2_Classification.ipynb` (WM-811K, Edge-Loc vs Loc) on a second semiconductor dataset, so the two can be compared like for like. The models, the search grids, the 60 / 20 / 20 split, the 5:1 cost ratio, the decision rule, the calibration step and the SHAP analysis are the same. Section 12 puts the two runs side by side.

Dataset: MixedWM38 mixed-type wafer maps (Kaggle `co1d7era/mixedtype-wafer-defect-datasets`).

## The decision this model supports
A **scratch** on a wafer map is a thin line of failing dies, usually from mechanical contact: a handling robot, a carrier, or a polishing step. It repeats until the tool is fixed. The model reads a wafer map and predicts **P(the wafer has a scratch)**. If the prediction is "scratch", the handling path is held for inspection.

In this dataset one wafer can carry several patterns at once (for example Center + Edge-Loc + Scratch), so the scratch has to be found among other defects. That makes it a harder question than the single-pattern one in the main project.

## Why the evaluation loss is not the misclassification rate
- **False negative** (a scratched wafer not flagged): the tool keeps scratching wafers until a routine check catches it.
- **False positive** (a clean wafer flagged): one unnecessary hold and inspection.

As in the main project, a miss is assumed to cost **5 times** a false alarm, and section 8 varies the ratio from 1 to 10.

$$\text{evaluation loss} = \frac{5 \cdot FN + 1 \cdot FP}{n}$$

## What differs from the main project, and why
| Difference | Reason |
|---|---|
| 9 features, not 10: `n_dies` is left out | In MixedWM38 the die count gives the label away (scratch wafers have 2,061–2,065 dies; the others range from 1,843 to 2,234). The screening notebook found this. |
| 1,024 exact duplicate maps are dropped | 97.7% of them were scratch wafers; copies in both training and validation inflate AUC. |
| Splits are stratified random, not by lot | MixedWM38 has no lot information. Each wafer is its own group, so the same code runs. |
| The class balance is artificial | The dataset's authors balanced the classes, partly with GAN-generated maps. The 49% scratch rate is not a fab's real rate, so the cost figures are not comparable in level, only in pattern. |


## 1. Setup and data

In [1]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
from IPython.display import display, Markdown

from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso, LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss, confusion_matrix
from sklearn.base import clone
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")
SEED = 42
C_FN, C_FP = 5.0, 1.0            # evaluation loss: a missed scratch costs 5 unnecessary tool holds
N_BOOT = 2000                    # bootstrap resamples (wafers)

DATA = "data/mixedwm38_scratch_features.csv"
MAPS = "data/mixedwm38_scratch_maps.npz"
FIG = "figures_scratch"; os.makedirs(FIG, exist_ok=True)

# Plot style: glassbox / no scratch = blue, blackbox / Scratch = orange; solid = log loss, dashed = L2
BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, INK2, MUTED, GRID = "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "axes.axisbelow": True, "legend.fontsize": 8, "figure.dpi": 110,
    "lines.linewidth": 2,
})
def savefig(name):
    plt.savefig(os.path.join(FIG, name), dpi=200, bbox_inches="tight")

df = pd.read_csv(DATA)
FEATURES = ["fail_rate", "center_fail_rate", "middle_fail_rate", "edge_fail_rate", "outer_ring_fail_rate",
            "n_fail_clusters", "largest_cluster_share", "largest_cluster_radius", "angular_concentration"]
TARGET = "scratch"
df["lot"] = df["wafer_id"]            # no lot information: each wafer is its own group
X, y, lots = df[FEATURES], df[TARGET], df["lot"]
print(f"{len(df):,} wafers, {len(FEATURES)} features, scratch rate {y.mean():.1%}")
df.head()

36,991 wafers, 9 features, scratch rate 48.7%


,wafer_id,fail_rate,center_fail_rate,middle_fail_rate,edge_fail_rate,outer_ring_fail_rate,n_fail_clusters,largest_cluster_share,largest_cluster_radius,angular_concentration,pattern,n_patterns,scratch,lot
0,0,0.300825,0.524324,0.220443,0.323308,0.509709,106,0.129549,0.884098,0.154518,Center + Edge-Loc + Scratch,3,1,0
1,1,0.300825,0.875676,0.210591,0.269737,0.368932,121,0.098981,0.222773,0.093790,Center + Edge-Loc + Scratch,3,1,1
2,2,0.327996,0.789189,0.274631,0.288534,0.419903,92,0.099951,0.246756,0.034598,Center + Edge-Loc + Scratch,3,1,2
3,3,0.277050,0.329730,0.184729,0.338346,0.514563,122,0.078117,0.888960,0.124899,Center + Edge-Loc + Scratch,3,1,3
4,4,0.248423,0.459459,0.250000,0.210526,0.266990,126,0.044153,0.205165,0.109107,Center + Edge-Loc + Scratch,3,1,4


**The data.** MixedWM38 has 38,015 wafer maps, all 52 × 52, each labelled with the patterns it contains (38 combinations of 8 basic patterns). `build_features_mixedwm38.py` drops the 1,024 exact duplicate maps, computes the same spatial features as in the main project, and saves 36,991 rows. The target is 1 if the wafer's label includes Scratch.

| Feature | Meaning |
|---|---|
| `fail_rate` | Share of dies that fail |
| `center_fail_rate`, `middle_fail_rate`, `edge_fail_rate`, `outer_ring_fail_rate` | Fail rate in the centre (radius < 0.3), middle (0.3–0.7), edge (≥ 0.7) and outermost ring (≥ 0.9); the radius is 1 at the outermost die |
| `n_fail_clusters` | Number of connected groups of failing dies |
| `largest_cluster_share` | Size of the largest cluster, as a share of all dies |
| `largest_cluster_radius` | Mean radius of the largest cluster (0 = centre, 1 = edge) |
| `angular_concentration` | 0 if failures are spread all around the wafer, 1 if they sit on one side |

None of these features describes a thin line directly. That is deliberate: the features are the ones from the main project, unchanged, so the comparison is fair. It also means a linear model has little to work with here.

In [2]:
# Data checks: all of these use features only, no labels
checks = pd.Series({
    "missing cells": int(df[FEATURES + [TARGET]].isna().sum().sum()),
    "duplicate feature rows": int(df.duplicated(subset=FEATURES).sum()),
    "duplicate feature rows with conflicting labels": int((df.groupby(FEATURES)[TARGET].nunique() > 1).sum()),
    "constant features": int((df[FEATURES].nunique() <= 1).sum()),
    "wafers with more than one pattern": int((df["n_patterns"] > 1).sum()),
    "scratched wafers that also carry another pattern": int(((df["scratch"] == 1) & (df["n_patterns"] > 1)).sum()),
})
display(checks.to_frame("value"))
display(df[FEATURES].describe().T[["mean", "std", "min", "50%", "max"]].round(3))

,value
missing cells,0
duplicate feature rows,0
duplicate feature rows with conflicting labels,0
constant features,0
wafers with more than one pattern,29000
scratched wafers that also carry another pattern,17000


,mean,std,min,50%,max
fail_rate,0.319,0.085,0.010,0.316,0.950
center_fail_rate,0.422,0.232,0.000,0.384,1.000
middle_fail_rate,0.289,0.116,0.006,0.252,1.000
edge_fail_rate,0.325,0.100,0.008,0.322,0.899
outer_ring_fail_rate,0.449,0.213,0.002,0.439,0.894
n_fail_clusters,102.699,30.012,1.000,105.000,181.000
largest_cluster_share,0.154,0.110,0.000,0.129,0.950
largest_cluster_radius,0.648,0.235,0.045,0.714,0.998
angular_concentration,0.115,0.068,0.000,0.105,0.566


## 2. Splits: train / validation / test
MixedWM38 has no lot information, so the split is stratified random (60 / 20 / 20). Each wafer is treated as its own group, which lets the code from the main project run unchanged. Exact duplicate maps were removed when the table was built, so no identical wafer sits in two splits.

- **Train:** fit the models; tune hyperparameters by 5-fold cross-validation.
- **Validation:** compare the four tuned models on AUC, choose one, set its threshold.
- **Test:** touched once, in section 10.

In [3]:
folds = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(X, y, lots))
test_idx, val_idx = folds[0][1], folds[1][1]
train_idx = np.setdiff1d(np.arange(len(df)), np.r_[test_idx, val_idx])
assert not set(lots.iloc[train_idx]) & set(lots.iloc[val_idx]) and not set(lots.iloc[train_idx]) & set(lots.iloc[test_idx])

X_tr, y_tr, g_tr = X.iloc[train_idx], y.iloc[train_idx], lots.iloc[train_idx]
X_va, y_va, g_va = X.iloc[val_idx], y.iloc[val_idx], lots.iloc[val_idx]
X_te, y_te, g_te = X.iloc[test_idx], y.iloc[test_idx], lots.iloc[test_idx]
display(pd.DataFrame({"wafers": [len(train_idx), len(val_idx), len(test_idx)],
                      "Scratch": [int(y_tr.sum()), int(y_va.sum()), int(y_te.sum())],
                      "Scratch rate": [f"{v:.1%}" for v in (y_tr.mean(), y_va.mean(), y_te.mean())]},
                     index=["train", "validation", "test"]))

,wafers,Scratch,Scratch rate
train,22195,10800,48.7%
validation,7398,3600,48.7%
test,7398,3600,48.7%


### A look at the training data
Only the training split is plotted, so nothing here leaks from validation or test.

In [4]:
fig, axes = plt.subplots(2, 5, figsize=(13, 4.6)); axes = axes.ravel()
for ax, c in zip(axes, FEATURES):
    v = X_tr[c]; hi = v.quantile(0.99) if c in ("n_dies", "n_fail_clusters") else v.max()
    bins = np.linspace(v.min(), hi, 31)
    ax.hist(v[y_tr == 0], bins=bins, density=True, histtype="step", lw=1.8, color=BLUE, label="No scratch")
    ax.hist(v[y_tr == 1], bins=bins, density=True, histtype="step", lw=1.8, color=ORANGE, label="Scratch")
    ax.set_title(c, fontsize=9); ax.set_yticks([])
for ax in axes[len(FEATURES):]: ax.axis('off')
axes[0].legend()
fig.suptitle("Feature distributions by class (training split)", fontweight="bold"); plt.tight_layout(); savefig("01_features_by_class.png"); plt.show()

corr = X_tr.corr()
hi_pairs = [(a, b, round(corr.loc[a, b], 2)) for i, a in enumerate(FEATURES) for b in FEATURES[i + 1:] if abs(corr.loc[a, b]) >= 0.85]
print("feature pairs with |r| ≥ 0.85 (training split):", hi_pairs)

feature pairs with |r| ≥ 0.85 (training split): [('fail_rate', 'largest_cluster_share', np.float64(0.87)), ('edge_fail_rate', 'outer_ring_fail_rate', np.float64(0.87))]


The zone fail rates are strongly correlated with the overall fail rate. That matters twice later: an L1 penalty is the right choice for the linear glassbox (it drops redundant features, which is why the slides prefer LASSO to ridge), and SHAP credit can be shared arbitrarily between correlated features (section 9).

## 3. The evaluation loss
A function of the decision threshold: flag a wafer as Scratch when its score is at least `t`.

In [5]:
def eval_loss(y_true, score, t, c_fn=C_FN, c_fp=C_FP):
    """Cost per wafer when wafers with score >= t are flagged as Scratch."""
    y_true = np.asarray(y_true); flag = np.asarray(score) >= t
    fn = np.sum((y_true == 1) & ~flag); fp = np.sum((y_true == 0) & flag)
    return (c_fn * fn + c_fp * fp) / len(y_true)

def best_threshold(y_true, score, c_fn=C_FN, c_fp=C_FP):
    """Exact minimiser over all thresholds that change a decision. Returns (threshold, cost)."""
    y_true = np.asarray(y_true); score = np.asarray(score)
    order = np.argsort(-score); s, yy = score[order], y_true[order]
    tp = np.cumsum(yy); fp = np.cumsum(1 - yy)                # flag the k highest scores
    cost = np.r_[c_fn * yy.sum(), c_fn * (yy.sum() - tp) + c_fp * fp] / len(yy)
    last = np.r_[True, np.r_[s[1:] != s[:-1], True]]          # only cut between distinct scores
    k = np.flatnonzero(last)[np.argmin(cost[last])]
    t = np.inf if k == 0 else (s[k - 1] if k == len(s) else (s[k - 1] + s[k]) / 2)
    return float(t), float(cost[k])

T_THEORY = C_FP / (C_FP + C_FN)
base = y_va.mean()
print(f"If the model's probabilities are calibrated, the cost-minimising threshold is C_FP / (C_FP + C_FN) = {T_THEORY:.3f}.")
print(f"Reference policies on validation: hold the tool for every wafer = {C_FP * (1 - base):.3f} per wafer; "
      f"never hold = {C_FN * base:.3f} per wafer.")

If the model's probabilities are calibrated, the cost-minimising threshold is C_FP / (C_FP + C_FN) = 0.167.
Reference policies on validation: hold the tool for every wafer = 0.513 per wafer; never hold = 2.433 per wafer.


## 4. Four models: glassbox and blackbox, each with L2 loss and with log loss
| | L2 loss (squared error on the 0/1 label) | Surrogate: binomial deviance (log loss) |
|---|---|---|
| **Glassbox:** L1-penalised linear model on standardised features | LASSO regression | L1 logistic regression |
| **Blackbox:** gradient-boosted trees | boosting with squared error | boosting with log loss |

Inside each row, the model structure and the search grid are the same and **only the training loss changes**, so the comparison isolates the loss.

**Tuning.** Each of the four is tuned separately by 5-fold cross-validated AUC inside the training split. The grids are deliberately small: one knob for the glassbox (penalty strength, 6 values) and two for the blackbox (tree depth × number of trees, 6 combinations). That's 24 configurations in total. The validation split plays no part in tuning, so it's only asked one question: which of four tuned models is best?

In [6]:
def score_of(model, X_):
    """One score per wafer: P(scratch) for log-loss models, the fitted value for L2 models."""
    return model.predict_proba(X_)[:, 1] if hasattr(model, "predict_proba") else model.predict(X_)

def auc_scorer(model, X_, y_):
    return roc_auc_score(y_, score_of(model, X_))

SPECS = {
    ("glassbox", "L2"): (make_pipeline(StandardScaler(), Lasso(max_iter=10000)),
                         {"lasso__alpha": [0.0003, 0.001, 0.003, 0.01, 0.03, 0.1]}),
    ("glassbox", "log loss"): (make_pipeline(StandardScaler(), LogisticRegression(penalty="l1", solver="liblinear", max_iter=2000, random_state=SEED)),
                               {"logisticregression__C": [0.003, 0.01, 0.03, 0.1, 1.0, 10.0]}),
    ("blackbox", "L2"): (HistGradientBoostingRegressor(loss="squared_error", learning_rate=0.1, random_state=SEED),
                         {"max_depth": [2, 3, 5], "max_iter": [100, 300]}),
    ("blackbox", "log loss"): (HistGradientBoostingClassifier(learning_rate=0.1, random_state=SEED),
                               {"max_depth": [2, 3, 5], "max_iter": [100, 300]}),
}
inner_cv = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(X_tr, y_tr, g_tr))

MODELS, SEARCH, rows = {}, {}, []
for key, (est, grid) in SPECS.items():
    gs = GridSearchCV(est, grid, scoring=auc_scorer, cv=inner_cv, n_jobs=-1, refit=True).fit(X_tr, y_tr)
    MODELS[key] = gs.best_estimator_; SEARCH[key] = gs
    rows.append({"model": key[0], "train loss": key[1], "configurations tried": len(gs.cv_results_["params"]),
                 "chosen": ", ".join(f"{k.split('__')[-1]}={v}" for k, v in gs.best_params_.items()),
                 "CV AUC (train split)": round(gs.best_score_, 4)})
tuning = pd.DataFrame(rows); display(tuning)
print("total configurations tried:", int(tuning["configurations tried"].sum()))

,model,train loss,configurations tried,chosen,CV AUC (train split)
0,glassbox,L2,6,alpha=0.0003,0.5944
1,glassbox,log loss,6,C=10.0,0.6459
2,blackbox,L2,6,"max_depth=5, max_iter=300",0.8698
3,blackbox,log loss,6,"max_depth=5, max_iter=300",0.8720


total configurations tried: 24


### How much does the glassbox penalty matter?
The L1 penalty is what keeps a linear model readable: a stronger penalty leaves fewer features. This table shows every penalty value tried for the L1 logistic model, with its cross-validated AUC on the training split and the number of features it keeps. It uses the training split only.

In [7]:
gs = SEARCH[("glassbox", "log loss")]
rows = []
for params, mean, sd in zip(gs.cv_results_["params"], gs.cv_results_["mean_test_score"], gs.cv_results_["std_test_score"]):
    m = clone(SPECS[("glassbox", "log loss")][0]).set_params(**params).fit(X_tr, y_tr)
    kept = [f for f, c in zip(FEATURES, m[-1].coef_.ravel()) if c != 0]
    rows.append({"C (larger = weaker penalty)": params["logisticregression__C"], "CV AUC": round(mean, 4), "sd over folds": round(sd, 4),
                 "features kept": len(kept), "dropped": ", ".join(f for f in FEATURES if f not in kept) or "none"})
display(pd.DataFrame(rows))

,C (larger = weaker penalty),CV AUC,sd over folds,features kept,dropped
0,0.003,0.5785,0.0109,2,"fail_rate, center_fail_rate, middle_fail_rate,..."
1,0.010,0.5784,0.0105,7,"center_fail_rate, largest_cluster_share"
2,0.030,0.5858,0.0094,8,fail_rate
3,0.100,0.5860,0.0093,8,fail_rate
4,1.000,0.6388,0.0071,9,none
5,10.000,0.6459,0.0067,9,none


## 5. Compare on validation AUC and decide
AUC is computed on the validation split, which no model has seen. The intervals come from a bootstrap that resamples wafers 2,000 times, and differences between models are computed on the same resamples (paired).

**Decision rule, fixed before looking at the results:**
1. *Within a family (L2 vs log loss):* take the higher validation AUC. If the difference isn't distinguishable from zero, take the log-loss version, because its output is a probability.
2. *Between families:* a blackbox has to earn its opacity. Choose it only if its validation AUC is distinguishably higher **and** that lowers the evaluation loss by at least 10%. Otherwise choose the glassbox.

In [8]:
S_va = {k: score_of(m, X_va) for k, m in MODELS.items()}
S_te = {k: score_of(m, X_te) for k, m in MODELS.items()}    # computed now, not looked at until section 10

def lot_bootstrap(groups, n_boot=N_BOOT, seed=SEED):
    """Row indices for n_boot resamples of whole groups."""
    groups = np.asarray(groups); rng = np.random.default_rng(seed)
    if len(pd.unique(groups)) == len(groups):                # every wafer is its own group: resample rows
        for _ in range(n_boot):
            yield rng.integers(0, len(groups), len(groups))
        return
    by_lot = [np.flatnonzero(groups == g) for g in pd.unique(groups)]
    for _ in range(n_boot):
        yield np.concatenate([by_lot[i] for i in rng.integers(0, len(by_lot), len(by_lot))])

def boot_stats(y_true, groups, fns):
    """Bootstrap each statistic in fns (name -> function of row indices) on the same resamples."""
    y_true = np.asarray(y_true); out = {k: [] for k in fns}
    for idx in lot_bootstrap(groups):
        if y_true[idx].min() == y_true[idx].max(): continue
        for k, f in fns.items(): out[k].append(f(idx))
    return {k: np.percentile(v, [2.5, 97.5]) for k, v in out.items()}

yv = np.asarray(y_va)
ci = boot_stats(yv, g_va, {k: (lambda idx, s=S_va[k]: roc_auc_score(yv[idx], s[idx])) for k in MODELS})
auc_tab = pd.DataFrame([{"model": k[0], "train loss": k[1], "validation AUC": roc_auc_score(yv, S_va[k]),
                         "95% CI low": ci[k][0], "95% CI high": ci[k][1]} for k in MODELS])
display(auc_tab.round(4))

def auc_diff(a, b):
    d = roc_auc_score(yv, S_va[a]) - roc_auc_score(yv, S_va[b])
    lo, hi = boot_stats(yv, g_va, {"d": lambda idx: roc_auc_score(yv[idx], S_va[a][idx]) - roc_auc_score(yv[idx], S_va[b][idx])})["d"]
    return d, lo, hi

# Rule 1: within each family
best = {}
for fam in ("glassbox", "blackbox"):
    d, lo, hi = auc_diff((fam, "log loss"), (fam, "L2"))
    best[fam] = (fam, "L2") if hi < 0 else (fam, "log loss")
    print(f"{fam}: log loss − L2 validation AUC = {d:+.4f} (95% CI {lo:+.4f} to {hi:+.4f}) -> keep {best[fam][1]}")

# Rule 2: between families
d, lo, hi = auc_diff(best["blackbox"], best["glassbox"])
cost_g = best_threshold(yv, S_va[best["glassbox"]])[1]; cost_b = best_threshold(yv, S_va[best["blackbox"]])[1]
saving = 1 - cost_b / cost_g
print(f"\nblackbox − glassbox validation AUC = {d:+.4f} (95% CI {lo:+.4f} to {hi:+.4f})")
print(f"evaluation loss at each model's best threshold: glassbox {cost_g:.4f}, blackbox {cost_b:.4f} per wafer "
      f"-> the blackbox lowers it by {saving:.1%}")
CHOSEN = best["blackbox"] if (lo > 0 and saving >= 0.10) else best["glassbox"]
OTHER = best["glassbox"] if CHOSEN[0] == "blackbox" else best["blackbox"]
display(Markdown(f"**Decision: {CHOSEN[0]}, trained with {CHOSEN[1]}.**"))

,model,train loss,validation AUC,95% CI low,95% CI high
0,glassbox,L2,0.5973,0.5843,0.6096
1,glassbox,log loss,0.6541,0.6412,0.6658
2,blackbox,L2,0.8788,0.8711,0.8861
3,blackbox,log loss,0.8810,0.8736,0.8882


glassbox: log loss − L2 validation AUC = +0.0568 (95% CI +0.0500 to +0.0644) -> keep log loss


blackbox: log loss − L2 validation AUC = +0.0022 (95% CI -0.0005 to +0.0049) -> keep log loss



blackbox − glassbox validation AUC = +0.2269 (95% CI +0.2148 to +0.2405)
evaluation loss at each model's best threshold: glassbox 0.5119, blackbox 0.3282 per wafer -> the blackbox lowers it by 35.9%


**Decision: blackbox, trained with log loss.**

In [9]:
STYLE = {("glassbox", "L2"): (BLUE, "--"), ("glassbox", "log loss"): (BLUE, "-"),
         ("blackbox", "L2"): (ORANGE, "--"), ("blackbox", "log loss"): (ORANGE, "-")}
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for k, s in S_va.items():
    fpr, tpr, _ = roc_curve(yv, s)
    for ax in axes: ax.plot(fpr, tpr, color=STYLE[k][0], ls=STYLE[k][1], lw=1.8, label=f"{k[0]}, {k[1]} (AUC {roc_auc_score(yv, s):.3f})")
axes[0].plot([0, 1], [0, 1], color=MUTED, lw=1, ls=":")
axes[0].set_title("ROC on validation"); axes[0].set_xlabel("false positive rate"); axes[0].set_ylabel("true positive rate"); axes[0].legend(loc="lower right")
axes[1].set_xlim(0, 0.3); axes[1].set_ylim(0.7, 1.0); axes[1].set_title("Zoom: the top-left corner"); axes[1].set_xlabel("false positive rate")
plt.tight_layout(); savefig("02_roc_validation.png"); plt.show()

fig, ax = plt.subplots(figsize=(6.5, 2.4))
for i, r in auc_tab.iloc[::-1].reset_index(drop=True).iterrows():
    k = (r["model"], r["train loss"])
    ax.plot([r["95% CI low"], r["95% CI high"]], [i, i], color=STYLE[k][0], lw=2, ls=STYLE[k][1])
    ax.plot(r["validation AUC"], i, "o", ms=8, color=STYLE[k][0], mec="#fcfcfb", mew=2)
    ax.text(r["95% CI high"] + 0.0008, i, f"{r['validation AUC']:.3f}", va="center", fontsize=8, color=INK2)
ax.set_yticks(range(4)); ax.set_yticklabels([f"{r['model']}, {r['train loss']}" for _, r in auc_tab.iloc[::-1].iterrows()])
ax.set_xlabel("validation AUC with 95% bootstrap interval"); ax.grid(axis="y", visible=False); ax.set_title("Four tuned models on validation")
plt.tight_layout(); savefig("03_auc_intervals.png"); plt.show()

### All four models, on more than one metric
The decision above uses validation AUC, as the brief requires. This table adds other views of the same four models. **Training AUC is shown only to make the brief's point:** it flatters the flexible model and says nothing about new wafers, which is why models are compared on validation.

In [10]:
from sklearn.metrics import average_precision_score, accuracy_score
ytr = np.asarray(y_tr); rows = []
for k, m in MODELS.items():
    s = S_va[k]; p = np.clip(s, 0, 1)
    rows.append({"model": k[0], "train loss": k[1],
                 "train AUC": roc_auc_score(ytr, score_of(m, X_tr)), "CV AUC (train)": SEARCH[k].best_score_,
                 "validation AUC": roc_auc_score(yv, s), "train − validation": roc_auc_score(ytr, score_of(m, X_tr)) - roc_auc_score(yv, s),
                 "average precision": average_precision_score(yv, s), "Brier": brier_score_loss(yv, p),
                 "accuracy at 0.5": accuracy_score(yv, s >= 0.5), "eval loss at 0.5": eval_loss(yv, s, 0.5),
                 "eval loss at best threshold": best_threshold(yv, s)[1]})
display(pd.DataFrame(rows).round(4))

,model,train loss,train AUC,CV AUC (train),validation AUC,train − validation,average precision,Brier,accuracy at 0.5,eval loss at 0.5,eval loss at best threshold
0,glassbox,L2,0.5956,0.5944,0.5973,-0.0018,0.5341,0.2423,0.5583,1.3209,0.4743
1,glassbox,log loss,0.6473,0.6459,0.6541,-0.0068,0.6238,0.2355,0.6021,1.2744,0.5119
2,blackbox,L2,0.9357,0.8698,0.8788,0.0569,0.8663,0.1425,0.7893,0.5714,0.3279
3,blackbox,log loss,0.9296,0.8720,0.8810,0.0486,0.8693,0.1406,0.7906,0.5954,0.3282


## 6. Threshold for the chosen model
For the chosen model we take the threshold that minimises the evaluation loss on validation, and compare it with the default 0.5.

In [11]:
s_va = S_va[CHOSEN]
T_STAR, cost_star = best_threshold(yv, s_va)
cost_05 = eval_loss(yv, s_va, 0.5)
ts = np.linspace(0.005, 0.995, 199)
curve = np.array([eval_loss(yv, s_va, t) for t in ts])
curve_mis = np.array([eval_loss(yv, s_va, t, 1, 1) for t in ts])
t_mis = best_threshold(yv, s_va, 1, 1)[0]

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(ts, curve, color=ORANGE, label=f"evaluation loss (FN = {C_FN:g}, FP = {C_FP:g})")
ax.plot(ts, curve_mis, color=BLUE, label="misclassification rate (FN = FP = 1)")
ax.axvline(T_STAR, color=INK2, lw=1, ls="--"); ax.text(T_STAR, curve.max() * 0.97, f" best: {T_STAR:.3f}", fontsize=8, color=INK2, va="top")
ax.axvline(0.5, color=MUTED, lw=1, ls=":"); ax.text(0.5, curve.max() * 0.97, " default 0.5", fontsize=8, color=MUTED, va="top")
ax.set_xlabel(f"threshold on the {CHOSEN[0]} score"); ax.set_ylabel("cost per wafer"); ax.set_title("Evaluation loss vs threshold (validation)"); ax.legend(loc="upper center")
plt.tight_layout(); savefig("04_threshold.png"); plt.show()

flag = s_va >= T_STAR
tn, fp, fn, tp = confusion_matrix(yv, flag).ravel()
print(f"best threshold = {T_STAR:.3f}: cost {cost_star:.4f} per wafer (FN {fn}, FP {fp}; recall {tp / (tp + fn):.1%}, tool holds {flag.mean():.1%} of wafers)")
tn5, fp5, fn5, tp5 = confusion_matrix(yv, s_va >= 0.5).ravel()
print(f"threshold 0.5:           cost {cost_05:.4f} per wafer (FN {fn5}, FP {fp5}; recall {tp5 / (tp5 + fn5):.1%})")
print(f"-> tuning the threshold lowers the evaluation loss by {1 - cost_star / cost_05:.1%}. "
      f"The threshold that minimises plain misclassification would be {t_mis:.3f}.")
t_lo, t_hi = np.percentile([best_threshold(yv[i], s_va[i])[0] for i in lot_bootstrap(g_va, 500)], [2.5, 97.5])
print(f"bootstrap 95% range of the best threshold: {t_lo:.3f} to {t_hi:.3f} (theory for a calibrated model: {T_THEORY:.3f})")

best threshold = 0.175: cost 0.3282 per wafer (FN 68, FP 2088; recall 98.1%, tool holds 76.0% of wafers)
threshold 0.5:           cost 0.5954 per wafer (FN 714, FP 835; recall 80.2%)
-> tuning the threshold lowers the evaluation loss by 44.9%. The threshold that minimises plain misclassification would be 0.488.
bootstrap 95% range of the best threshold: 0.162 to 0.239 (theory for a calibrated model: 0.167)


## 7. Calibration
A model is calibrated if, among wafers given a score near 0.3, about 30% are Scratch. We measure it on validation with a reliability diagram, the expected calibration error (ECE: the average gap between predicted and observed rates, over 10 bins) and the Brier score.

**The fix.** We fit a calibrator on **out-of-fold training scores** (5 folds, same hyperparameters), then apply it to validation. Fitting the calibrator on validation would reuse the data that also sets the threshold. Two calibrators are tried: Platt scaling (a one-feature logistic regression, two parameters) and isotonic regression (a monotone step function).

In [12]:
def ece(y_true, p, bins=10):
    y_true = np.asarray(y_true); p = np.clip(np.asarray(p, float), 0, 1)
    b = np.clip((p * bins).astype(int), 0, bins - 1)
    return float(sum((b == i).mean() * abs(y_true[b == i].mean() - p[b == i].mean()) for i in range(bins) if (b == i).any()))

def reliability(y_true, p, bins=10):
    y_true = np.asarray(y_true); p = np.clip(np.asarray(p, float), 0, 1)
    q = np.unique(np.quantile(p, np.linspace(0, 1, bins + 1)))               # equal-count bins
    b = np.clip(np.digitize(p, q[1:-1]), 0, len(q) - 2)
    return (np.array([p[b == i].mean() for i in range(len(q) - 1) if (b == i).any()]),
            np.array([y_true[b == i].mean() for i in range(len(q) - 1) if (b == i).any()]))

# Out-of-fold training scores for every model, with its tuned hyperparameters
OOF = {}
for k, m in MODELS.items():
    method = "predict_proba" if hasattr(m, "predict_proba") else "predict"
    o = cross_val_predict(clone(m), X_tr, y_tr, cv=inner_cv, method=method, n_jobs=-1)
    OOF[k] = o[:, 1] if o.ndim == 2 else o

def to_feature(k, s):
    """Platt input: the logit for probability models, the raw score for L2 models."""
    s = np.asarray(s, float)
    if k[1] == "L2": return s.reshape(-1, 1)
    s = np.clip(s, 1e-6, 1 - 1e-6); return np.log(s / (1 - s)).reshape(-1, 1)

CAL = {}
for k in MODELS:
    platt = LogisticRegression(C=1e6, max_iter=1000).fit(to_feature(k, OOF[k]), y_tr)
    iso = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(OOF[k], y_tr)
    CAL[k] = {"Platt": (lambda s, k=k, m=platt: m.predict_proba(to_feature(k, s))[:, 1]),
              "isotonic": (lambda s, m=iso: m.predict(np.asarray(s, float)))}

rows = []
for k in MODELS:
    raw = S_va[k]
    row = {"model": k[0], "train loss": k[1], "scores outside [0, 1]": f"{np.mean((raw < 0) | (raw > 1)):.1%}",
           "ECE raw": ece(yv, raw), "ECE Platt": ece(yv, CAL[k]["Platt"](raw)), "ECE isotonic": ece(yv, CAL[k]["isotonic"](raw)),
           "Brier raw": brier_score_loss(yv, np.clip(raw, 0, 1)), "Brier Platt": brier_score_loss(yv, CAL[k]["Platt"](raw)),
           "Brier isotonic": brier_score_loss(yv, CAL[k]["isotonic"](raw))}
    rows.append(row)
cal_tab = pd.DataFrame(rows); display(cal_tab.round(4))

def ece_ci(p):
    return np.percentile([ece(yv[i], p[i]) for i in lot_bootstrap(g_va, 500)], [2.5, 97.5])
raw = S_va[CHOSEN]
for name, p in [("raw", raw), ("Platt", CAL[CHOSEN]["Platt"](raw)), ("isotonic", CAL[CHOSEN]["isotonic"](raw))]:
    lo, hi = ece_ci(p); print(f"chosen model, {name:8s}: ECE {ece(yv, p):.4f} (95% CI {lo:.4f} to {hi:.4f})")

,model,train loss,"scores outside [0, 1]",ECE raw,ECE Platt,ECE isotonic,Brier raw,Brier Platt,Brier isotonic
0,glassbox,L2,0.0%,0.0476,0.0384,0.0041,0.2423,0.2418,0.2340
1,glassbox,log loss,0.0%,0.0435,0.0435,0.0136,0.2355,0.2354,0.2290
2,blackbox,L2,10.4%,0.0240,0.0263,0.0104,0.1425,0.1424,0.1418
3,blackbox,log loss,0.0%,0.0132,0.0094,0.0117,0.1406,0.1404,0.1405


chosen model, raw     : ECE 0.0132 (95% CI 0.0115 to 0.0239)


chosen model, Platt   : ECE 0.0094 (95% CI 0.0087 to 0.0211)


chosen model, isotonic: ECE 0.0117 (95% CI 0.0103 to 0.0230)


In [13]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
ax = axes[0]; ax.plot([0, 1], [0, 1], color=MUTED, lw=1, ls=":")
for k in MODELS:
    mp, ob = reliability(yv, S_va[k]); ax.plot(mp, ob, marker="o", ms=4, lw=1.6, color=STYLE[k][0], ls=STYLE[k][1], label=f"{k[0]}, {k[1]}")
ax.set_title("Before calibration: all four models"); ax.set_xlabel("mean predicted probability (clipped to 0–1)"); ax.set_ylabel("observed scratch rate"); ax.legend()
ax = axes[1]; ax.plot([0, 1], [0, 1], color=MUTED, lw=1, ls=":")
col = STYLE[CHOSEN][0]
for name, p, ls, mk in [("raw", raw, "-", "o"), ("Platt", CAL[CHOSEN]["Platt"](raw), "--", "s"), ("isotonic", CAL[CHOSEN]["isotonic"](raw), ":", "^")]:
    mp, ob = reliability(yv, p); ax.plot(mp, ob, marker=mk, ms=5, lw=1.6, color=col if name == "raw" else INK2, ls=ls, label=f"{name} (ECE {ece(yv, p):.3f})")
ax.set_title(f"Chosen model ({CHOSEN[0]}, {CHOSEN[1]}): before and after"); ax.set_xlabel("mean predicted probability"); ax.legend()
plt.tight_layout(); savefig("05_calibration.png"); plt.show()

# Pick the calibrator by validation Brier score (a proper scoring rule); keep raw if neither helps
briers = {"raw": brier_score_loss(yv, np.clip(raw, 0, 1)), "Platt": brier_score_loss(yv, CAL[CHOSEN]["Platt"](raw)),
          "isotonic": brier_score_loss(yv, CAL[CHOSEN]["isotonic"](raw))}
CALIB = min(briers, key=briers.get)
calibrate = (lambda s: np.clip(np.asarray(s, float), 0, 1)) if CALIB == "raw" else CAL[CHOSEN][CALIB]
p_va = calibrate(raw)
T_CAL, cost_cal = best_threshold(yv, p_va)
print(f"calibrator with the lowest validation Brier score: {CALIB}  {({k: round(v, 4) for k, v in briers.items()})}")
print(f"AUC before {roc_auc_score(yv, raw):.4f}, after {roc_auc_score(yv, p_va):.4f} (a monotone map can only change AUC through ties)")
print(f"best threshold on calibrated probabilities = {T_CAL:.3f} (cost {cost_cal:.4f}); theory says {T_THEORY:.3f}, "
      f"which costs {eval_loss(yv, p_va, T_THEORY):.4f} on validation")

calibrator with the lowest validation Brier score: Platt  {'raw': 0.1406, 'Platt': 0.1404, 'isotonic': 0.1405}
AUC before 0.8810, after 0.8810 (a monotone map can only change AUC through ties)
best threshold on calibrated probabilities = 0.155 (cost 0.3282); theory says 0.167, which costs 0.3317 on validation


**Why this matters for the threshold.** If the probabilities are calibrated, we don't need to search for a threshold at all: decision theory gives it as C_FP / (C_FP + C_FN) = 0.167, and it updates by formula when the costs change. The last line above checks how close the searched threshold on calibrated probabilities is to that value, and what using the formula would cost.

**The final decision rule** is: calibrate the chosen model's score, then hold the tool when the calibrated probability is at least the threshold found on validation.

## 8. How sensitive is the decision to the cost ratio?
The 5:1 ratio is our assumption. Here the ratio runs from 1 (plain misclassification) to 10. For each ratio we show the best threshold and the resulting cost, for the chosen model and for the best model of the other family.

In [14]:
ratios = np.arange(1, 11)
p_other = CAL[OTHER][CALIB](S_va[OTHER]) if CALIB != "raw" else np.clip(S_va[OTHER], 0, 1)
sens = pd.DataFrame({"FN:FP ratio": ratios,
                     "best threshold (chosen)": [best_threshold(yv, p_va, r, 1)[0] for r in ratios],
                     "theory 1/(1+ratio)": 1 / (1 + ratios),
                     f"cost, chosen ({CHOSEN[0]})": [best_threshold(yv, p_va, r, 1)[1] for r in ratios],
                     f"cost, other ({OTHER[0]})": [best_threshold(yv, p_other, r, 1)[1] for r in ratios],
                     "cost, hold every wafer": [(1 - yv.mean())] * len(ratios)})
display(sens.round(3))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].plot(ratios, sens["best threshold (chosen)"], "o-", color=STYLE[CHOSEN][0], label="best threshold on validation")
axes[0].plot(ratios, sens["theory 1/(1+ratio)"], color=MUTED, ls=":", label="theory for a calibrated model")
axes[0].set_xlabel("cost of a missed scratch, in unnecessary tool holds"); axes[0].set_ylabel("threshold"); axes[0].set_title("Threshold vs cost ratio"); axes[0].legend()
axes[1].plot(ratios, sens[f"cost, chosen ({CHOSEN[0]})"], "o-", color=STYLE[CHOSEN][0], label=f"chosen: {CHOSEN[0]}")
axes[1].plot(ratios, sens[f"cost, other ({OTHER[0]})"], "s-", color=STYLE[OTHER][0], label=f"other: {OTHER[0]}")
axes[1].axvline(C_FN / C_FP, color=INK2, lw=1, ls="--"); axes[1].text(C_FN / C_FP, axes[1].get_ylim()[0], " our assumption", fontsize=8, color=INK2, va="bottom")
axes[1].set_xlabel("cost of a missed scratch, in unnecessary tool holds"); axes[1].set_ylabel("cost per wafer at the best threshold"); axes[1].set_title("Evaluation loss vs cost ratio"); axes[1].legend()
plt.tight_layout(); savefig("06_cost_sensitivity.png"); plt.show()

,FN:FP ratio,best threshold (chosen),theory 1/(1+ratio),"cost, chosen (blackbox)","cost, other (glassbox)","cost, hold every wafer"
0,1,0.485,0.500,0.208,0.389,0.513
1,2,0.332,0.333,0.267,0.474,0.513
2,3,0.252,0.250,0.298,0.495,0.513
3,4,0.200,0.200,0.316,0.511,0.513
4,5,0.155,0.167,0.328,0.512,0.513
5,6,0.155,0.143,0.337,0.512,0.513
6,7,0.155,0.125,0.347,0.512,0.513
7,8,0.142,0.111,0.356,0.512,0.513
8,9,0.084,0.100,0.363,0.512,0.513
9,10,0.084,0.091,0.368,0.512,0.513


## 9. Interpretability: tree SHAP on the tree-based blackbox, global and local
Whichever model we chose, the brief asks for tree SHAP on a tree-based blackbox. We explain the **gradient-boosted trees trained with log loss**. SHAP values are in log-odds: a value of +1 for a feature means that feature moved this wafer's log-odds of Scratch up by 1 from the average prediction.

We also put the glassbox's own explanation (its coefficients) next to the global SHAP ranking. If two very different models lean on the same features, that's evidence the signal is in the data, not an artefact of one model.

In [15]:
import shap
gbm = MODELS[("blackbox", "log loss")]
explainer = shap.TreeExplainer(gbm)
sv = np.asarray(explainer.shap_values(X_va))
if sv.ndim == 3: sv = sv[..., 1]
base_value = float(np.ravel(explainer.expected_value)[-1])
p_gbm = S_va[("blackbox", "log loss")]
logit = np.log(np.clip(p_gbm, 1e-9, 1 - 1e-9) / (1 - np.clip(p_gbm, 1e-9, 1 - 1e-9)))
print(f"additivity check: max |base + sum(SHAP) − model log-odds| = {np.abs(base_value + sv.sum(1) - logit).max():.2e}")

glob = pd.Series(np.abs(sv).mean(0), index=FEATURES).sort_values()
lin = MODELS[("glassbox", "log loss")]
coef = pd.Series(lin[-1].coef_.ravel(), index=FEATURES)
rho = spearmanr(glob[FEATURES], coef.abs()[FEATURES]).correlation

seq = LinearSegmentedColormap.from_list("seq", ["#cde2fb", "#2a78d6", "#0d366b"])
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), gridspec_kw={"width_ratios": [1, 1.5, 1]})
axes[0].barh(glob.index, glob.values, color=ORANGE, height=0.6); axes[0].grid(axis="y", visible=False)
axes[0].set_title("Global: mean |SHAP| (blackbox)"); axes[0].set_xlabel("mean |SHAP| (log-odds)")
rng = np.random.default_rng(SEED)
for i, f in enumerate(glob.index):
    j = FEATURES.index(f); v = X_va[f].rank(pct=True).values
    sc = axes[1].scatter(sv[:, j], i + rng.uniform(-0.3, 0.3, len(sv)), c=v, cmap=seq, s=5, alpha=0.7, linewidths=0)
axes[1].axvline(0, color=MUTED, lw=1); axes[1].set_yticks(range(len(glob))); axes[1].set_yticklabels(glob.index); axes[1].grid(axis="y", visible=False)
axes[1].set_title("Global: every validation wafer"); axes[1].set_xlabel("SHAP value (log-odds; right = towards Scratch)")
cb = fig.colorbar(sc, ax=axes[1], pad=0.01, shrink=0.8); cb.set_label("feature value (percentile)", fontsize=8); cb.set_ticks([0, 1]); cb.set_ticklabels(["low", "high"])
c2 = coef[glob.index]
axes[2].barh(c2.index, c2.values, color=BLUE, height=0.6); axes[2].axvline(0, color=MUTED, lw=1); axes[2].grid(axis="y", visible=False)
axes[2].set_yticklabels([]); axes[2].set_title("Glassbox: L1 logistic coefficients"); axes[2].set_xlabel("coefficient per standard deviation (log-odds)")
plt.tight_layout(); savefig("07_shap_global.png"); plt.show()
print("global ranking (mean |SHAP|):", ", ".join(f"{f} {v:.2f}" for f, v in glob.iloc[::-1].items()))
print(f"features the L1 glassbox dropped: {list(coef.index[coef == 0]) or 'none'}")
print(f"rank agreement between mean |SHAP| and |glassbox coefficient| (Spearman): {rho:.2f}")

additivity check: max |base + sum(SHAP) − model log-odds| = 2.18e-12


global ranking (mean |SHAP|): edge_fail_rate 1.18, outer_ring_fail_rate 1.04, n_fail_clusters 0.70, largest_cluster_radius 0.67, angular_concentration 0.41, middle_fail_rate 0.35, largest_cluster_share 0.32, fail_rate 0.26, center_fail_rate 0.16
features the L1 glassbox dropped: none
rank agreement between mean |SHAP| and |glassbox coefficient| (Spearman): -0.15


In [16]:
top = glob.index[-1]; second = glob.index[-2]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for ax, f in zip(axes, (top, second)):
    j = FEATURES.index(f)
    ax.scatter(X_va[f][yv == 0], sv[yv == 0, j], s=6, color=BLUE, alpha=0.5, linewidths=0, label="No scratch")
    ax.scatter(X_va[f][yv == 1], sv[yv == 1, j], s=6, color=ORANGE, alpha=0.5, linewidths=0, label="Scratch")
    ax.axhline(0, color=MUTED, lw=1); ax.set_xlabel(f); ax.set_ylabel("SHAP value (log-odds)"); ax.set_title(f"How {f} moves the prediction")
    if f in ("n_dies", "n_fail_clusters"): ax.set_xlim(X_va[f].min(), X_va[f].quantile(0.99))
axes[0].legend(markerscale=2.5)
plt.tight_layout(); savefig("08_shap_dependence.png"); plt.show()

### Local explanations, next to the wafer itself
Three validation wafers, chosen by rule (not by hand):
1. the scratched wafer the blackbox is **most sure** about
2. the unscratched wafer it is most sure about
3. its **costliest mistake**: the scratched wafer with the lowest predicted probability (a false negative, which costs 5)

Each row shows the wafer map and the SHAP values that add up to that wafer's prediction.

In [17]:
z = np.load(MAPS); map_row = {r: i for i, r in enumerate(z["wafer_id"])}
wafer_cmap = ListedColormap(["#fcfcfb", "#cde2fb", ORANGE])        # no die, passing die, failing die

def wafer_map(row_id):
    m = z["maps"][map_row[row_id]]
    keep_r = np.flatnonzero((m != 255).any(1)); keep_c = np.flatnonzero((m != 255).any(0))
    return m[: keep_r.max() + 1, : keep_c.max() + 1]

pos, neg = np.flatnonzero(yv == 1), np.flatnonzero(yv == 0)
picks = [("Most confident scratch", pos[np.argmax(p_gbm[pos])]),
         ("Most confident no-scratch", neg[np.argmin(p_gbm[neg])]),
         ("Costliest mistake: a scratched wafer scored as clean", pos[np.argmin(p_gbm[pos])])]

fig, axes = plt.subplots(len(picks), 2, figsize=(11, 3.3 * len(picks)), gridspec_kw={"width_ratios": [1, 2.6]})
for (title, i), (axm, axs) in zip(picks, axes):
    row = df.iloc[val_idx[i]]
    axm.imshow(wafer_map(row["wafer_id"]), cmap=wafer_cmap, vmin=0, vmax=2, interpolation="nearest"); axm.axis("off")
    axm.set_title(f"{title}\nlabel: {row['pattern']}", fontsize=9)
    contrib = pd.Series(sv[i], index=FEATURES); contrib = contrib.reindex(contrib.abs().sort_values().index)
    labels = [f"{f} = {X_va.iloc[i][f]:.3g}" for f in contrib.index]
    axs.barh(labels, contrib.values, color=[ORANGE if v > 0 else BLUE for v in contrib.values], height=0.6)
    axs.axvline(0, color=MUTED, lw=1); axs.grid(axis="y", visible=False)
    axs.set_title(f"base {base_value:+.2f} + SHAP {sv[i].sum():+.2f} = log-odds {logit[i]:+.2f}  →  P(scratch) = {p_gbm[i]:.3f}", fontsize=9)
    axs.set_xlabel("SHAP value (log-odds; orange pushes towards Scratch, blue towards no scratch)")
plt.tight_layout(); savefig("09_shap_local.png"); plt.show()
p_glass = S_va[("glassbox", "log loss")]
for title, i in picks:
    print(f"{title}: blackbox P = {p_gbm[i]:.3f}, glassbox P = {p_glass[i]:.3f}")

Most confident scratch: blackbox P = 1.000, glassbox P = 0.556
Most confident no-scratch: blackbox P = 0.000, glassbox P = 0.059
Costliest mistake: a scratched wafer scored as clean: blackbox P = 0.013, glassbox P = 0.525


**How far to trust these plots.** The slides warn that every explanation rests on strong assumptions. Three apply here:
- **Correlated features.** The zone fail rates move together. SHAP splits credit among correlated features in a way that depends on how the trees happened to split, so the ranking *among the fail-rate features* isn't reliable.
- **SHAP explains the model, not the fab.** A large SHAP value says the model relies on a feature. It doesn't say that changing the feature would change the defect.
- **Indirect features.** No feature measures a line. The model infers a scratch from side effects (more small clusters, a different spread of failures), so its explanations are about those side effects, not about the scratch itself.

## 10. Final check on the test split (used once)
Everything is frozen: the chosen model, its calibrator and its threshold. The test split has not been used for any decision above.

In [18]:
yt = np.asarray(y_te)
p_te = calibrate(S_te[CHOSEN])
auc_te = roc_auc_score(yt, p_te)
lo, hi = boot_stats(yt, g_te, {"a": lambda idx: roc_auc_score(yt[idx], p_te[idx])})["a"]
flag = p_te >= T_CAL
tn, fp, fn, tp = confusion_matrix(yt, flag).ravel()
cost_te = eval_loss(yt, p_te, T_CAL)
c_lo, c_hi = np.percentile([eval_loss(yt[i], p_te[i], T_CAL) for i in lot_bootstrap(g_te)], [2.5, 97.5])
final = pd.DataFrame({
    "validation": [roc_auc_score(yv, p_va), cost_cal, eval_loss(yv, p_va, 0.5), ece(yv, p_va), C_FP * (1 - yv.mean()), C_FN * yv.mean()],
    "test": [auc_te, cost_te, eval_loss(yt, p_te, 0.5), ece(yt, p_te), C_FP * (1 - yt.mean()), C_FN * yt.mean()],
}, index=["AUC", f"evaluation loss at threshold {T_CAL:.3f}", "evaluation loss at threshold 0.5", "ECE (calibrated)",
          "evaluation loss: hold every wafer", "evaluation loss: never hold"])
display(final.round(4))
print(f"test AUC 95% CI: {lo:.4f} to {hi:.4f}; test evaluation loss 95% CI: {c_lo:.4f} to {c_hi:.4f}")
print(f"test confusion at threshold {T_CAL:.3f}: TP {tp}, FN {fn}, FP {fp}, TN {tn} (recall {tp / (tp + fn):.1%}, precision {tp / (tp + fp):.1%}, tool holds {flag.mean():.1%} of wafers)")
print(f"for reference only (not used for any decision): test AUC of all four = "
      + ", ".join(f"{k[0]}/{k[1]} {roc_auc_score(yt, S_te[k]):.4f}" for k in MODELS))

summary = {"chosen_model": list(CHOSEN), "calibrator": CALIB, "threshold": round(T_CAL, 4), "cost_ratio_fn_fp": [C_FN, C_FP],
           "validation_auc": {f"{k[0]}|{k[1]}": round(float(roc_auc_score(yv, S_va[k])), 4) for k in MODELS},
           "test_auc_chosen": round(float(auc_te), 4), "test_eval_loss": round(float(cost_te), 4),
           "test_eval_loss_at_0.5": round(float(eval_loss(yt, p_te, 0.5)), 4), "configurations_tried": int(tuning["configurations tried"].sum())}
json.dump(summary, open("results_summary_scratch.json", "w"), indent=2); print(json.dumps(summary, indent=2))

,validation,test
AUC,0.8810,0.8818
evaluation loss at threshold 0.155,0.3282,0.3305
evaluation loss at threshold 0.5,0.5985,0.5835
ECE (calibrated),0.0094,0.0133
evaluation loss: hold every wafer,0.5134,0.5134
evaluation loss: never hold,2.4331,2.4331


test AUC 95% CI: 0.8742 to 0.8888; test evaluation loss 95% CI: 0.3163 to 0.3446
test confusion at threshold 0.155: TP 3543, FN 57, FP 2160, TN 1638 (recall 98.4%, precision 62.1%, tool holds 77.1% of wafers)
for reference only (not used for any decision): test AUC of all four = glassbox/L2 0.5853, glassbox/log loss 0.6306, blackbox/L2 0.8786, blackbox/log loss 0.8818
{
  "chosen_model": [
    "blackbox",
    "log loss"
  ],
  "calibrator": "Platt",
  "threshold": 0.1553,
  "cost_ratio_fn_fp": [
    5.0,
    1.0
  ],
  "validation_auc": {
    "glassbox|L2": 0.5973,
    "glassbox|log loss": 0.6541,
    "blackbox|L2": 0.8788,
    "blackbox|log loss": 0.881
  },
  "test_auc_chosen": 0.8818,
  "test_eval_loss": 0.3305,
  "test_eval_loss_at_0.5": 0.5835,
  "configurations_tried": 24
}


### Post-hoc audit of the model choice (reference only)
This is a check on our own decision, done after everything was frozen. The model we did **not** choose is put through the same steps (same calibrator rule, threshold set on validation) and scored on test. Nothing here changes the decision: picking a model after seeing its test score would make the test result meaningless.

In [19]:
raw_o = S_va[OTHER]
br = {"raw": brier_score_loss(yv, np.clip(raw_o, 0, 1)), "Platt": brier_score_loss(yv, CAL[OTHER]["Platt"](raw_o)),
      "isotonic": brier_score_loss(yv, CAL[OTHER]["isotonic"](raw_o))}
cal_o = min(br, key=br.get)
calibrate_o = (lambda s: np.clip(np.asarray(s, float), 0, 1)) if cal_o == "raw" else CAL[OTHER][cal_o]
t_o, cost_o_val = best_threshold(yv, calibrate_o(raw_o))
p_te_o = calibrate_o(S_te[OTHER])
cost_o_te = eval_loss(yt, p_te_o, t_o)
d_lo, d_hi = np.percentile([eval_loss(yt[i], p_te_o[i], t_o) - eval_loss(yt[i], p_te[i], T_CAL) for i in lot_bootstrap(g_te)], [2.5, 97.5])
tn_o, fp_o, fn_o, tp_o = confusion_matrix(yt, p_te_o >= t_o).ravel()
audit = pd.DataFrame({
    f"chosen: {CHOSEN[0]}, {CHOSEN[1]}": [CALIB, T_CAL, cost_cal, cost_te, auc_te, accuracy_score(yt, p_te >= T_CAL), fn, fp],
    f"not chosen: {OTHER[0]}, {OTHER[1]}": [cal_o, t_o, cost_o_val, cost_o_te, roc_auc_score(yt, p_te_o), accuracy_score(yt, p_te_o >= t_o), fn_o, fp_o],
}, index=["calibrator", "threshold (set on validation)", "validation evaluation loss", "test evaluation loss", "test AUC", "test accuracy", "test FN", "test FP"])
display(audit)
print(f"test evaluation loss, not chosen − chosen = {cost_o_te - cost_te:+.4f} (95% CI {d_lo:+.4f} to {d_hi:+.4f})")

,"chosen: blackbox, log loss","not chosen: glassbox, log loss"
calibrator,Platt,isotonic
threshold (set on validation),0.155296,0.151515
validation evaluation loss,0.328197,0.512706
test evaluation loss,0.330495,0.513382
test AUC,0.881847,0.628342
test accuracy,0.700324,0.489321
test FN,57,5
test FP,2160,3773


test evaluation loss, not chosen − chosen = +0.1829 (95% CI +0.1684 to +0.1972)


## 11. Findings, and what to flag

### What we found
1. **The blackbox is chosen here, by the same rule that chose the glassbox in the main project.** Validation AUC is 0.881 for the boosted trees and 0.654 for the L1 logistic model (difference +0.227, 95% CI +0.215 to +0.241). The blackbox lowers the evaluation loss by 36% (0.328 vs 0.512 per wafer), well past the 10% it needs to earn its opacity.
2. **The glassbox is no better than holding every wafer.** Its best evaluation loss (0.512) equals the cost of flagging everything (0.513). A linear model on zone fail rates can't see a thin line among other defects.
3. **The training loss matters for the glassbox, barely for the blackbox.** Log loss beats L2 by +0.057 AUC for the glassbox (CI +0.050 to +0.064). For the blackbox the difference is +0.002 and the interval includes zero (−0.0005 to +0.005); the tie goes to log loss because its output is a probability. The L2 blackbox puts 10% of its scores outside 0–1.
4. **The threshold matters more than in the main project.** The cost-minimising threshold is 0.175 on the raw score. It cuts the validation loss by 45% against 0.5 (0.595 → 0.328), by catching 646 more scratched wafers for 1,253 more false alarms.
5. **Calibration was already good, and the threshold lands on theory.** The blackbox's calibration error is 0.013 raw and 0.009 after Platt scaling. On calibrated probabilities the best threshold is 0.155 against the theoretical 0.167, and for cost ratios 1 to 4 the searched threshold matches the formula to within 0.02.
6. **The choice holds for every cost ratio from 1 to 10.**
7. **Test result (used once).** AUC 0.882 (95% CI 0.874 to 0.889), evaluation loss 0.330 per wafer (CI 0.316 to 0.345), recall 98.4%, precision 62.1%, accuracy 70.0%. Validation and test agree (0.328 vs 0.330): with 7,398 wafers in each, there is little optimiser's curse.
8. **The audit agrees with the choice.** The glassbox scores 0.513 on test, 0.183 worse than the blackbox (CI +0.168 to +0.197).
9. **Even the chosen model saves little in absolute terms.** At the best threshold it holds the tool for 77% of wafers. The loss (0.330) is 36% below holding every wafer (0.513).

### Flags
| # | Flag | What it means |
|---|---|---|
| 1 | **The model does not look at the scratch.** SHAP ranks `edge_fail_rate` and `outer_ring_fail_rate` first. The wafer it is most sure has a scratch (Donut + Edge-Ring + Loc + Scratch) shows no visible line in its map; the prediction comes from the edge fail rates. | The model has learned which mixes of patterns tend to come with the Scratch label in this dataset. That may be a property of how the mixed maps were built, not of scratches. It would not transfer to a fab. |
| 2 | **The class balance is artificial.** 49% of wafers are scratched because the authors balanced the classes, partly with GAN-generated maps. | The loss figures can't be compared in level with the main project or with a real fab. |
| 3 | **`n_dies` had to be deleted and 1,024 duplicate maps dropped** before the task was honest. | With them, the screening notebook measured AUC 0.917; cleaned, 0.88. |
| 4 | **The blackbox search ended at the edge of its grid** (depth 5, 300 trees). | A larger grid might do better. We kept the grid identical to the main project for comparability. |
| 5 | **Splits are random, not by lot.** | The dataset has no lot information. Near-duplicate generated maps could still sit in different splits. |
| 6 | **The features were not designed for scratches.** | Kept identical to the main project on purpose. A line detector would be the obvious next feature. |
| 7 | **Kaggle usability rating not verified.** | Check the dataset page. |

## 12. Side by side with the main project
Same pipeline, same grids, same 5:1 cost ratio, same decision rule.

| | Main project: WM-811K, Edge-Loc vs Loc | This run: MixedWM38, Scratch vs not |
|---|---|---|
| Data | Real fab maps; 8,782 wafers; 59% positive | Real + GAN-generated maps; 36,991 wafers; 49% positive (balanced by the authors) |
| Features | 10 | 9 (`n_dies` deleted as a shortcut) |
| Split | By lot | Stratified random (no lot information) |
| Validation AUC, glassbox L2 / log loss | 0.958 / 0.966 | 0.597 / 0.654 |
| Validation AUC, blackbox L2 / log loss | 0.966 / 0.972 | 0.879 / 0.881 |
| Log loss − L2, glassbox | +0.008 | +0.057 |
| Log loss − L2, blackbox | +0.007 | +0.002 (interval includes zero) |
| Blackbox − glassbox AUC | +0.006 (CI +0.0001 to +0.012) | +0.227 (CI +0.215 to +0.241) |
| Evaluation loss at best threshold, glassbox / blackbox | 0.143 / 0.149 | 0.512 / 0.328 |
| **Chosen model** | **Glassbox** (the blackbox didn't lower the cost) | **Blackbox** (36% lower cost) |
| Threshold (raw score) and saving vs 0.5 | 0.28; −27% | 0.175; −45% |
| Calibration error before → after | 0.031 → 0.021 (isotonic) | 0.013 → 0.009 (Platt) |
| Calibrated threshold vs theory (0.167) | 0.22 | 0.155 |
| Top SHAP feature | `largest_cluster_radius` | `edge_fail_rate` |
| Does SHAP match the physics? | Yes: is the cluster at the edge? | No: the model reads side effects, not the line |
| Test AUC | 0.958 | 0.882 |
| Test evaluation loss (vs holding every wafer) | 0.181 (0.409): −56% | 0.330 (0.513): −36% |
| Validation → test loss | 0.143 → 0.181 | 0.328 → 0.330 |
| Post-hoc audit | The unchosen blackbox did somewhat better on test (not distinguishable) | The unchosen glassbox did far worse |

**What the comparison shows**
- **The same rule gives opposite model choices**, and for a clear reason: where a linear model captures the signal (edge vs interior), the glassbox is enough; where it can't (a thin line among other defects), the blackbox is worth its opacity.
- **Log loss beats L2 in every case, or ties.** It never loses.
- **The threshold step pays off in both**, and more where the model is weaker.
- **Better calibration puts the searched threshold closer to the theoretical one** (0.155 vs 0.167 here; 0.22 vs 0.167 in the main project).
- **A larger validation split removes most of the optimiser's curse** (7,398 wafers here against 1,756).

**Which to present.** The main project remains the stronger one to present: real data, a model whose explanation matches the physics, and no shortcut features. This run has the more dramatic glassbox-vs-blackbox result, but flag 1 means its model is likely learning how the dataset was assembled. It works well as a second case that shows the method reaching a different, justified decision.
